## About Dataset

Context

A small subset of dataset of product reviews from Amazon Kindle Store category.

Content

5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset.

Columns

- asin - ID of the product, like B000FA64PK
- helpful - helpfulness rating of the review - example: 2/3.
- overall - rating of the product.
- reviewText - text of the review (heading).
- reviewTime - time of the review (raw).
- reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
- reviewerName - name of the reviewer.
- summary - summary of the review (description).
- unixReviewTime - unix timestamp.

Acknowledgements

This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

Inspiration

- Sentiment analysis on reviews.
- Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
- Fake reviews/ outliers.
- Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
- Any other interesting analysis.

### Best Practices
1. Preprocessing And Cleaning
2. Train Test Split
3. BOW, TF-IDF, Word2Vec
4. Train ML Algorithms

In [15]:
# Load the dataset
import pandas as pd
data=pd.read_csv('kindle_review.csv')

In [16]:
data.head()

,Unnamed: 0,asin,helpful,overall,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,B000F83SZQ,"[0, 0]",5,I enjoy vintage books and movies so I enjoyed ...,"05 5, 2014",A1F6404F1VG29J,Avidreader,Nice vintage story,1399248000
1,1,B000F83SZQ,"[2, 2]",4,This book is a reissue of an old one; the auth...,"01 6, 2014",AN0N05A9LIJEQ,critters,Different...,1388966400
2,2,B000F83SZQ,"[2, 2]",4,This was a fairly interesting read. It had ol...,"04 4, 2014",A795DMNCJILA6,dot,Oldie,1396569600
3,3,B000F83SZQ,"[1, 1]",5,I'd never read any of the Amy Brewster mysteri...,"02 19, 2014",A1FV0SX13TWVXQ,"Elaine H. Turley ""Montana Songbird""",I really liked it.,1392768000
4,4,B000F83SZQ,"[0, 1]",4,"If you like period pieces - clothing, lingo, y...","03 19, 2014",A3SPTOKDG7WBLN,Father Dowling Fan,Period Mystery,1395187200


In [17]:
data=data[['reviewText','overall']]
data.head()

,reviewText,overall
0,I enjoy vintage books and movies so I enjoyed ...,5
1,This book is a reissue of an old one; the auth...,4
2,This was a fairly interesting read. It had ol...,4
3,I'd never read any of the Amy Brewster mysteri...,5
4,"If you like period pieces - clothing, lingo, y...",4


In [18]:
data.shape

(3960, 2)

In [19]:
# Missing Values
data.isnull().sum()

reviewText    0
overall       0
dtype: int64

In [20]:
data['overall'].unique()

array([5, 4, 3, 2, 1])

In [21]:
data['overall'].value_counts()

overall
5    1778
4    1206
3     551
2     244
1     181
Name: count, dtype: int64

In [22]:
# Preprocessing and Cleaning

In [23]:
# positive review 1 and negative review is 0
data['overall']=data['overall'].apply(lambda x:0 if x<3 else 1)

In [24]:
data['overall'].value_counts()

overall
1    3535
0     425
Name: count, dtype: int64

In [25]:
# 1. Lower All the cases
data['reviewText']=data['reviewText'].str.lower()

In [26]:
data.head()

,reviewText,overall
0,i enjoy vintage books and movies so i enjoyed ...,1
1,this book is a reissue of an old one; the auth...,1
2,this was a fairly interesting read. it had ol...,1
3,i'd never read any of the amy brewster mysteri...,1
4,"if you like period pieces - clothing, lingo, y...",1


In [27]:
import re
import nltk
from nltk.corpus import stopwords
from bs4 import BeautifulSoup

In [28]:
# Removing special characters
data['reviewText']=data['reviewText'].apply(lambda x: re.sub('[^a-z A-Z 0-9-]+','',x))
# Remove the stopwords
data['reviewText']=data['reviewText'].apply(lambda x: ' '.join([y for y in x.split() if y not in stopwords.words('english')]))
# Remove url
data['reviewText']=data['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?','',str(x)))
# Remove html tags
data['reviewText']=data['reviewText'].apply(lambda x: BeautifulSoup(str(x), 'html.parser').get_text())
# Remove any additional spaces
data['reviewText']=data['reviewText'].apply(lambda x: " ".join(x.split()))

In [29]:
data.head()

,reviewText,overall
0,enjoy vintage books movies enjoyed reading boo...,1
1,book reissue old one author born 1910 era say ...,1
2,fairly interesting read old- style terminology...,1
3,id never read amy brewster mysteries one reall...,1
4,like period pieces - clothing lingo enjoy myst...,1


In [30]:
# Lemmatizer
from nltk.stem import WordNetLemmatizer

In [31]:
lemmatizer=WordNetLemmatizer()

In [32]:
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [33]:
data['reviewText']=data['reviewText'].apply(lambda x: lemmatize_words(x))

In [34]:
data.head()

,reviewText,overall
0,enjoy vintage book movie enjoyed reading book ...,1
1,book reissue old one author born 1910 era say ...,1
2,fairly interesting read old- style terminology...,1
3,id never read amy brewster mystery one really ...,1
4,like period piece - clothing lingo enjoy myste...,1


In [36]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(data['reviewText'],data['overall'],test_size=0.20)

In [45]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow=bow.fit_transform(X_train).toarray()
X_test_bow=bow.transform(X_test).toarray()

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train).toarray()
X_test_tfidf=tfidf.transform(X_test).toarray()

In [47]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(3168, 18674))

In [48]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)

In [49]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

In [50]:
y_pred_bow=nb_model_bow.predict(X_test_bow)
y_pred_tfidf=nb_model_tfidf.predict(X_test_tfidf)

In [51]:
print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))

BOW accuracy:  0.773989898989899


In [52]:
confusion_matrix(y_test,y_pred_bow)

array([[ 18,  76],
       [103, 595]])

In [56]:
print(classification_report(y_test, y_pred_bow))

              precision    recall  f1-score   support

           0       0.15      0.19      0.17        94
           1       0.89      0.85      0.87       698

    accuracy                           0.77       792
   macro avg       0.52      0.52      0.52       792
weighted avg       0.80      0.77      0.79       792



In [53]:
print("TF-IDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))

TF-IDF accuracy:  0.773989898989899


In [54]:
confusion_matrix(y_test,y_pred_tfidf)

array([[ 18,  76],
       [103, 595]])

In [55]:
print(classification_report(y_test, y_pred_tfidf))

              precision    recall  f1-score   support

           0       0.15      0.19      0.17        94
           1       0.89      0.85      0.87       698

    accuracy                           0.77       792
   macro avg       0.52      0.52      0.52       792
weighted avg       0.80      0.77      0.79       792

